In [ ]:
import time
import json
import os
import sys
import math
sys.path.append(os.path.dirname(os.getcwd()))
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm
from matplotlib.colors import Normalize
from PIL import Image

from pc import PS
from modules import ADC,DAC,CHIP
from command import CMD,CmdData,Packet
from command.singleCmdInfo import *

from util import plot_v_cond,plot_cond,show_crossbar,DataLoader

from network.layer import Layer

In [ ]:
chip=CHIP(PS(host="192.168.1.10", port = 7, debug=0),init=True)
chip.set_device_cfg(deviceType=0,IsNew32=False)
chip.adc.set_gap(adc_cs_gap=90,adc_first_gap=100,adc_last_gap=100)
chip.adc.set_gain_resistor(big_resistance=10e3,small_resistance=200)
chip.clk_manager.set_cyc(10, 10,delay3=50)
chip.add_compiler("../compiler/code/")
chip.compensation.initop("../chip_data/chip6_/")

In [ ]:
v,c,r = chip.read4(crossbar=np.ones((10,10)),row_index=None,col_index=None,read_voltage=0.1,tg=5,gain=1,sub_base=True,from_row=True,split_type=0,row_type=0,col_type=0)
c0 = chip.compensation.compensation_point(r,from_row=True,return_type=0,)

v,c,r = chip.read4(crossbar=np.ones((10,10)),row_index=[i for i in range(10)],col_index=[i for i in range(10)],
                   read_voltage=0.1,tg=5,gain=1,sub_base=True,from_row=True,split_type=4,row_type=0,col_type=0)


compensation_para:dict={
"value":0.875,
"real_mean":550,
"odd_offset":0,
"even_offset":0,
"odd_mult":1,
"even_mult":1,
"all_mult":1,
"add_wire":False
}
cond = chip.compensation.compensation_forward(rows_index,res[2],from_row=True,return_type=0,compensation_para=compensation_para)
print(c.shape)
plt.plot(c[startpos:10])
plt.show()

### 1. 单列线性补偿

In [ ]:
v,c,r = chip.read4(crossbar=np.ones((256,256)),row_index=None,col_index=None,read_voltage=0.1,tg=5,gain=1,sub_base=True,from_row=True,split_type=0,row_type=0,col_type=0)
c = chip.compensation.compensation_point(r,from_row=True,return_type=0,)
# plot_cond(c,vmax=1000)

In [ ]:
compensation_para:dict={
"value":0.875,
"real_mean":550,
"odd_offset":0,
"even_offset":0,
"odd_mult":1,
"even_mult":1,
"all_mult":1,
"add_wire":False
}

c = chip.compensation.compensation_point(r,from_row=True,return_type=0,)

In [ ]:
col = 10
select_num,total_num = 50,200


random_times = 100
real_res = np.zeros(random_times)
expected_res = np.zeros(random_times)
for i in range(random_times):
    rows_index = np.random.choice(total_num, select_num, replace=False)
    res = chip.read4(crossbar=np.ones((256,256)),row_index=rows_index,col_index=[col],read_voltage=0.1,tg=5,gain=3,sub_base=True,from_row=True,split_type=3,row_type=0,col_type=0)
    cond = chip.compensation.compensation_forward(rows_index,res[2],from_row=True,return_type=0,compensation_para=compensation_para)
    # print(np.max(res[num]))
    expected_res[i]=np.sum(c[rows_index,col])
    real_res[i]=cond[col]

In [ ]:
coefficients = np.polyfit(real_res, expected_res, 1)
print("拟合系数:", coefficients[num],coefficients[1])

# 拟合的线性函数
def linear_fit(x):
    return coefficients[num] * x + coefficients[1]

# 计算拟合的 y 值
y_fit = linear_fit(real_res)

plt.figure(figsize=(12, 5))

plt.subplot(1,2,1)
plt.title(f"random row count = {select_num}")
plt.plot(expected_res,label = "expected output")
plt.plot(real_res,label = "actual output")
plt.plot(y_fit,label = "fit output")
plt.ylabel("uS")
plt.xlabel("times")
plt.legend()

plt.subplot(1,2,2)
plt.plot(np.zeros(random_times))
plt.plot(real_res/expected_res-1,label = "actual output/expected output")
plt.plot(y_fit/expected_res-1,label = "fit output/expected output")
plt.ylabel("100%")
plt.xlabel("times")
plt.legend()
plt.title(f"random row count = {select_num}")
plt.show()

### 2. 矩阵求逆补偿

In [ ]:
v,c,r = chip.read4(crossbar=np.ones((256,256)),row_index=None,col_index=None,read_voltage=0.1,tg=5,gain=1,sub_base=True,from_row=True,split_type=0,row_type=0,col_type=0)
c = chip.compensation.compensation_point(r,from_row=True,return_type=0)

In [ ]:
col_index = [i for i in range(10,20)]
random_times = 5000
real_res = np.zeros((random_times,len(col_index)))
expected_res = np.zeros((random_times,len(col_index)))
for i in range(random_times):
    # 这次的输入是多少行
    row_num = np.random.choice(total_num, 1, replace=False)[num]
    while row_num ==0 :
        row_num = np.random.choice(total_num, 1, replace=False)[num]
    # 随机选row_num行作为输入
    rows_index = np.random.choice(total_num, row_num, replace=False)

    res = chip.read4(crossbar=np.ones((256,256)),row_index=rows_index,col_index=col_index,read_voltage=0.1,tg=5,gain=3,sub_base=True,from_row=True,split_type=4,row_type=0,col_type=0)
    cond = chip.compensation.compensation_forward(rows_index,res[2],from_row=True,return_type=0,compensation_para=compensation_para)
    # print(np.max(res[num]))
    pos = np.ix_(rows_index,col_index)
    # print(np.sum(c[pos],axis=0).shape)
    expected_res[i,:]=np.sum(c[pos],axis=0)
    real_res[i,:]=cond[col_index]

In [ ]:
rows_index = [i for i in range(10,20)]
random_times = 5000
real_res = np.zeros((random_times,len(rows_index)))
expected_res = np.zeros((random_times,len(rows_index)))
for i in range(random_times):
    # 这次的输入是多少行
    row_num = np.random.choice(total_num, 1, replace=False)[num]
    while row_num ==0 :
        row_num = np.random.choice(total_num, 1, replace=False)[num]
    # 随机选row_num行作为输入
    col_index = np.random.choice(total_num, row_num, replace=False)

    res = chip.read4(crossbar=np.ones((256,256)),row_index=rows_index,col_index=col_index,read_voltage=0.1,tg=5,gain=3,sub_base=True,from_row=True,split_type=4,row_type=0,col_type=0)
    cond = chip.compensation.compensation_forward(rows_index,res[2],from_row=True,return_type=0,compensation_para=compensation_para)
    # print(np.max(res[num]))
    pos = np.ix_(rows_index,col_index)
    # print(np.sum(c[pos],axis=0).shape)
    expected_res[i,:]=np.sum(c[pos],axis=1)
    real_res[i,:]=cond[rows_index]

In [ ]:
A_pinv = np.linalg.pinv(real_res)
W_pinv = np.dot(A_pinv,expected_res)
expected_res2 = np.dot(expected_res ,W_pinv)

data = expected_res2/expected_res-1
x = [i for i in range(10)]

In [ ]:
data = real_res/expected_res-1
data[data<-2] = -2
x = [i for i in range(10)]
for i in range(5000):
    plt.scatter(x, data[i], color='blue')
plt.plot(np.zeros(10))
# 添加标题和标签
plt.title("Y\'/Y")
plt.ylabel("100%")
plt.xlabel("col")

# 添加图例
# plt.legend()

# 显示图表
plt.show()

In [ ]:
print(np.sum(np.abs(data)>0.1)/50000)

### 3. 线阻补偿

In [ ]:
v,c_expected_from_row,r = chip.read4(crossbar=np.ones((256,256)),row_index=None,col_index=None,read_voltage=0.1,tg=5,gain=1,sub_base=True,from_row=True,split_type=0,row_type=0,col_type=0)
c_expected_from_row = chip.compensation.compensation_point(r,from_row=True,return_type=0)

v,c_expected_from_col,r = chip.read4(crossbar=np.ones((256,256)),row_index=None,col_index=None,read_voltage=0.1,tg=5,gain=1,sub_base=True,from_row=False,split_type=0,row_type=0,col_type=0)
c_expected_from_col = chip.compensation.compensation_point(r,from_row=False,return_type=0)

In [ ]:
compensation_para:dict={
"value":0.5,
"real_mean":550,
"odd_offset":0,
"even_offset":0,
"odd_mult":1,
"even_mult":1,
"all_mult":1,
"add_wire":False
}

In [ ]:
print(chip.compensation.col_r_out_crossbar)

In [ ]:
chip.compensation.initop("../chip_data/chip6_/")

In [ ]:
# for i in range(256):
#     print(chip.compensation.col_r_out[i],chip.compensation.col_value[i])

In [ ]:
# 10列，两路TIA并行
col_index = [i for i in range(256)]
# 存储数据
real_res = np.zeros((256,256))
real_cond_res = np.zeros((256,256))
expected_res = np.zeros((256,256))

# chip.compensation.col_value = 1
# chip.compensation.col_offset=0
for i in range(101):
    row_index = [j for j in range(i+1)]

    _,cond,r = chip.read4(crossbar=np.ones((256,256)),row_index=row_index,col_index=col_index,read_voltage=0.1,tg=5,gain=3,sub_base=True,from_row=True,split_type=3,row_type=0,col_type=0)
    cond = chip.compensation.compensation_forward(row_index,r,from_row=True,return_type=0,compensation_para=None)
    pos = np.ix_(row_index,col_index)
    expected_res[i,:]=np.sum(c_expected_from_row[pos],axis=0)
    real_res[i,:]=r
    real_cond_res[i,:] = cond[col_index]

In [ ]:
startpos,endpos = 0,101
num = 0
for num in range(101):
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    plt.plot(real_cond_res[startpos:endpos,num].flatten(),label = "actual output")
    plt.plot(expected_res[startpos:endpos,num].flatten(),label = "expected output")
    plt.title(f"col={col_index[num]}")
    plt.legend()
    # plt.show()
    plt.subplot(1,2,2)
    plt.plot(real_cond_res[startpos:endpos,num].flatten()/expected_res[startpos:endpos,num].flatten(),label = "actual output/expected output")
    plt.plot(expected_res[startpos:endpos,num].flatten()/expected_res[startpos:endpos,num].flatten(),label = "expected output/expected output")
    plt.title(f"col={col_index[num]}")
    plt.legend()
    plt.show()

In [ ]:
from scipy.stats import norm
from scipy.optimize import curve_fit

In [ ]:
offset_col = chip.compensation.col_offset
r_out_col = chip.compensation.col_r_out
print(offset_col[73])
print(r_out_col[73])

In [ ]:
r_wire = chip.compensation.r_wire
r_out_col = chip.compensation.col_r_out
offset_col = chip.compensation.col_offset
def compensation_value_from_row(x,value,col):
    """
        计算补偿
    """
    rows = len(x)
    ans = np.zeros(rows)
    for i in range(rows):
        sum_rw = (255-i)*r_wire if col%2==0 else 0
        r_out = r_out_col[col]
        real_r = real_res[i,col]*1e3 -r_out -sum_rw + offset_col[col]  - r_wire*(i**value)
        ans[i]=1/real_r*1e6
    return ans/expected_res[:rows,col]

cnt = 101

ans = np.zeros(256)

for i in range(256):
    params, covariance = curve_fit(lambda x,value:compensation_value_from_row(x,value,i),[i for i in range(cnt)], np.ones(cnt), p0=[0.5],bounds=([0], [1]))
    ans[i] = params[0]
print(ans)

[1.00000000e+00 8.87082098e-01 9.03097472e-01 9.20297618e-01
 9.15159977e-01 9.50818634e-01 9.05149795e-01 9.40173190e-01
 8.77745237e-01 9.49404291e-01 8.92864898e-01 9.51313216e-01
 9.14988195e-01 9.37964753e-01 9.11565336e-01 9.29270155e-01
 9.04244501e-01 9.73402610e-01 8.95872679e-01 9.56589861e-01
 8.97582814e-01 9.60234798e-01 8.85738535e-01 9.37708141e-01
 8.91720817e-01 9.38137150e-01 8.99471195e-01 8.98464292e-01
 8.85576445e-01 9.35881416e-01 8.77026874e-01 9.56406819e-01
 8.84789374e-01 9.09415817e-01 8.68611477e-01 9.35347778e-01
 9.03687953e-01 9.23564594e-01 9.25852171e-01 9.54116997e-01
 8.92549104e-01 9.33880056e-01 8.89752674e-01 9.42070894e-01
 8.78731706e-01 9.50190001e-01 8.78311647e-01 9.68000939e-01
 8.64803620e-01 9.27056239e-01 9.28915843e-01 9.28502295e-01
 9.10189854e-01 9.22342242e-01 8.95013642e-01 9.59649382e-01
 8.86970340e-01 9.74669151e-01 8.92503708e-01 9.43245803e-01
 8.86312313e-01 9.23977819e-01 8.71818507e-01 9.27623750e-01
 8.93290988e-01 9.572404

In [ ]:
np.save("../chip_data/chip6_/col_value.npy",ans)
np.save("../chip_data/chip6_/row_value.npy",ans)

In [ ]:
r_wire = chip.compensation.r_wire
r_out_col = chip.compensation.col_r_out
def compensation_offset_from_row(x,offset,col):
    """
        计算补偿
    """
    rows = len(x)
    ans = np.zeros(rows)
    for i in range(rows):
        sum_rw = (255-i)*r_wire if col%2==0 else 0
        r_out = r_out_col[col]
        real_r = real_res[i,col]*1e3 -r_out -sum_rw + offset
        ans[i]=1/real_r*1e6
    return ans/expected_res[:rows,col]

cnt = 31
ans = np.zeros(256)

for i in range(256):
    params, covariance = curve_fit(lambda x,offset:compensation_offset_from_row(x,offset,i),[i for i in range(cnt)], np.ones(cnt), p0=[0],bounds=([-10], [10]))
    ans[i] = params[0]

In [ ]:
np.save("../chip_data/chip6_/col_offset.npy",ans)
np.save("../chip_data/chip6_/row_offset.npy",ans)
print(np.max(ans),np.min(ans))
plt.hist(ans)
plt.show()

In [ ]:
startpos,endpos = 0,101
num = 0
for num in range(10):
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    plt.plot(real_cond_res[startpos:endpos,num].flatten(),label = "actual output")
    plt.plot(expected_res[startpos:endpos,num].flatten(),label = "expected output")
    plt.title(f"col={col_index[num]}")
    plt.legend()
    # plt.show()
    plt.subplot(1,2,2)
    plt.plot(real_cond_res[startpos:endpos,num].flatten()/expected_res[startpos:endpos,num].flatten(),label = "actual output/expected output")
    plt.plot(expected_res[startpos:endpos,num].flatten()/expected_res[startpos:endpos,num].flatten(),label = "expected output/expected output")
    plt.title(f"col={col_index[num]}")
    plt.legend()
    plt.show()

In [ ]:
# 10列，两路TIA并行
row_index = [10,11]
# 存储数据
real_res = np.zeros((256,len(row_index)))
expected_res = np.zeros((256,len(row_index)))

for i in range(256):
    col_index = [j for j in range(i+1)]

    _,cond,_ = chip.read4(crossbar=np.ones((256,256)),row_index=row_index,col_index=col_index,read_voltage=0.1,tg=5,gain=3,sub_base=True,from_row=False,split_type=4,row_type=0,col_type=0)

    pos = np.ix_(row_index,col_index)
    expected_res[i,:]=np.sum(c_expected_from_col[pos],axis=1)
    real_res[i,:]=cond[row_index]

In [ ]:
pos = 4
plt.plot(real_res[startpos:,num].flatten(),label = "actual output")
plt.plot(expected_res[startpos:,num].flatten(),label = "expected output")
plt.legend()
plt.show()

plt.plot(real_res[startpos:,1].flatten(),label = "actual output")
plt.plot(expected_res[startpos:,1].flatten(),label = "expected output")
plt.legend()
plt.show()

In [ ]:
compensation_para:dict={
"value":0.5,
"real_mean":550,
"odd_offset":0,
"even_offset":0,
"odd_mult":1,
"even_mult":1,
"all_mult":1,
"add_wire":True
}

In [ ]:
# 10列，两路TIA并行
col_index = [10,11]
random_times = 200
select_num,total_num = 50,256
# 存储数据
real_res = np.zeros((random_times,len(col_index)))
expected_res = np.zeros((random_times,len(col_index)))

for i in range(random_times):
    rows_index = np.random.choice(total_num, select_num, replace=False)

    res = chip.read4(crossbar=np.ones((256,256)),row_index=rows_index,col_index=col_index,read_voltage=0.1,tg=5,gain=3,sub_base=True,from_row=True,split_type=4,row_type=0,col_type=0)
    cond = chip.compensation.compensation_forward(rows_index,res[2],from_row=True,return_type=0,compensation_para=compensation_para)
    # cond = res[1]
    # print(cond)
    pos = np.ix_(rows_index,col_index)
    expected_res[i,:]=np.sum(c[pos],axis=0)
    real_res[i,:]=cond[col_index]

In [ ]:
pos = 4
plt.plot(real_res[startpos:,num].flatten(),label = "actual output")
plt.plot(expected_res[startpos:,num].flatten(),label = "expected output")
plt.legend()
plt.show()

plt.plot(real_res[startpos:,1].flatten(),label = "actual output")
plt.plot(expected_res[startpos:,1].flatten(),label = "expected output")
plt.legend()
plt.show()

In [ ]:
pos = 4
plt.plot(real_res[startpos:,num].flatten(),label = "actual output")
plt.plot(expected_res[startpos:,num].flatten(),label = "expected output")
plt.legend()
plt.show()

plt.plot(real_res[startpos:,1].flatten(),label = "actual output")
plt.plot(expected_res[startpos:,1].flatten(),label = "expected output")
plt.legend()
plt.show()